In [1]:
from load_data import load_raw_data
from clean_data import clean_data
from split_data import split_data
from evaluate_model import evaluate_model
from config import RANDOM_STATE
from build_pipeline import objective,build_pipeline
import optuna
import logging

In [2]:
logging.basicConfig(
    level=logging.INFO,format="%(asctime)s | %(levelname)s | %(name)s | %(message)s")

In [3]:
# Loading raw data
dataset=load_raw_data()

2026-06-05 20:05:55,374 | INFO | load_data | Raw dataset loaded.


In [4]:
# Cleaning data
dataset_cleaned=clean_data(dataset)

2026-06-05 20:05:55,378 | INFO | clean_data | Dropped customerID.
2026-06-05 20:05:55,380 | INFO | clean_data | Converted TotalCharges to numeric.
2026-06-05 20:05:55,384 | INFO | clean_data | Rows containing NaN before cleaning: 11
2026-06-05 20:05:55,391 | INFO | clean_data | Removed 11 rows with invalid TotalCharges. Remaining rows with NaN: 0.


In [5]:
# Splitting data
X_train, X_test, y_train, y_test=split_data(dataset_cleaned, random_state=RANDOM_STATE)

2026-06-05 20:05:55,399 | INFO | split_data | Data split into train/test set with 0.8/0.2 proportion and random_state=0.


### XGBoost

In [6]:
# Creating optuna study for XGBoost
study_XGB = optuna.create_study(direction="maximize")

[I 2026-06-05 20:06:00,197] A new study created in memory with name: no-name-ef13e01b-64b4-4709-8815-553a54c6ca1d


In [7]:
# Optimizing optuna study for XGBoost
study_XGB.optimize(lambda trial: objective(trial, X_train, y_train, model='XGBoost'),n_trials=100)

[I 2026-06-05 20:06:02,495] Trial 0 finished with value: 0.5341264233433726 and parameters: {'xgb_learning_rate': 0.0029467794204662955, 'xgb_max_depth': 4, 'xgb_min_child_weight': 5.187537192495929, 'xgb_gamma': 9.106036416095488, 'xgb_subsample': 0.6171353703792833, 'xgb_colsample_bytree': 0.9679151958217522, 'xgb_colsample_bylevel': 0.8261902427872474, 'xgb_reg_alpha': 0.051481572417310294, 'xgb_reg_lambda': 1.805340127855906e-08, 'xgb_scale_pos_weight': 19.839410701160006}. Best is trial 0 with value: 0.5341264233433726.
[I 2026-06-05 20:06:04,315] Trial 1 finished with value: 0.5839741177936352 and parameters: {'xgb_learning_rate': 0.002201858224985671, 'xgb_max_depth': 5, 'xgb_min_child_weight': 15.7443768546974, 'xgb_gamma': 8.04470269631125, 'xgb_subsample': 0.9101751586335226, 'xgb_colsample_bytree': 0.9730787776966543, 'xgb_colsample_bylevel': 0.8346933844327675, 'xgb_reg_alpha': 6.426857320695259, 'xgb_reg_lambda': 4.6431911090244e-06, 'xgb_scale_pos_weight': 5.6220818533463

In [8]:
print("Best score for XGB:", study_XGB.best_value)
print("Best params for XGB:", study_XGB.best_params)

Best score for XGB: 0.6384184537733159
Best params for XGB: {'xgb_learning_rate': 0.027642086939155982, 'xgb_max_depth': 5, 'xgb_min_child_weight': 3.219811850106856, 'xgb_gamma': 8.029768158431493, 'xgb_subsample': 0.669461795447187, 'xgb_colsample_bytree': 0.7662725936654277, 'xgb_colsample_bylevel': 0.696577876506165, 'xgb_reg_alpha': 0.015877402461164602, 'xgb_reg_lambda': 1.4915579938968426, 'xgb_scale_pos_weight': 1.8448148048233282}


In [9]:
# Fitting Logistic Regression pipeline with the best trial
pipe_XGB=build_pipeline(trial=study_XGB.best_trial, model= 'XGBoost')
fitted_pipe_XGB=pipe_XGB.fit(X_train, y_train)

### LightGBM

In [6]:
# Creating optuna study for LightGBM
study_LGBM = optuna.create_study(direction="maximize")

[I 2026-06-05 00:03:31,856] A new study created in memory with name: no-name-6868c481-b484-4626-ad23-9fd1d3fe3dce


In [7]:
# Optimizing optuna study for LightGBM
study_LGBM.optimize(lambda trial: objective(trial, X_train, y_train, model='LightGBM'),n_trials=100)

[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.189896 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.243208 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.149981 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

...

/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-06-05 00:08:20,458] Trial 2 finished with value: 0.5973863389848756 and parameters: {'lgbm_learning_rate': 0.025135320877542732, 'lgbm_max_depth': 10, 'lgbm_num_leaves': 503, 'lgbm_min_child_samples': 71, 'lgbm_subsample': 0.9678657543702787, 'lgbm_colsample_bytree': 0.9249253494381747, 'lgbm_reg_alpha': 5.266486392238566, 'lgbm_reg_lambda

[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000227 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

[W 2026-06-05 00:08:28,702] Trial 3 failed with parameters: {'lgbm_learning_rate': 0.015865336698271375, 'lgbm_max_depth': 7, 'lgbm_num_leaves': 84, 'lgbm_min_child_samples': 41, 'lgbm_subsample': 0.8305370358277393, 'lgbm_colsample_bytree': 0.744038854750514, 'lgbm_reg_alpha': 0.04898642704240225, 'lgbm_reg_lambda': 0.18064302986657754} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/joblib/parallel.py", line 1682, in _get_outputs
    yield from self._retrieve()
  File "/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/joblib/parallel.py", line 1800, in _retrieve
    time.sleep(0.01)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/cassia/miniconda3/envs/churnenv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func

In [ ]:
print("Best score for LGBM:", study_LGBM.best_value)
print("Best params for LGBM:", study_LGBM.best_params)

In [ ]:
# Fitting LightGBM pipeline with the best trial
pipe_LGBM=build_pipeline(trial=study_LR.best_trial, model= 'LightGBM')
fitted_pipe_LGBM=pipe_LGBM.fit(X_train, y_train)

### Logistic Regression

In [10]:
# Creating optuna study for Logistic Regression
study_LR = optuna.create_study(direction="maximize")

[I 2026-06-05 20:06:54,597] A new study created in memory with name: no-name-e045509a-b397-4de9-bf6e-e763e18b9315


In [11]:
# Optimizing optuna study for Logistic Regression
study_LR.optimize(lambda trial: objective(trial, X_train, y_train, model='Logistic Regression'),n_trials=100)

[I 2026-06-05 20:06:54,732] Trial 0 finished with value: 0.6276025515262618 and parameters: {'solver': 'saga', 'l1_ratio': 0.016293899384187793, 'C': 0.025898256507153064}. Best is trial 0 with value: 0.6276025515262618.
[I 2026-06-05 20:06:56,076] Trial 1 finished with value: 0.6307580501654348 and parameters: {'solver': 'saga', 'l1_ratio': 0.19775164196857598, 'C': 3.7753021584988287}. Best is trial 1 with value: 0.6307580501654348.
[I 2026-06-05 20:06:57,912] Trial 2 finished with value: 0.6314416638549953 and parameters: {'solver': 'saga', 'l1_ratio': 0.34829606289451265, 'C': 18.100978669883432}. Best is trial 2 with value: 0.6314416638549953.
[I 2026-06-05 20:06:58,301] Trial 3 finished with value: 0.6321202525997326 and parameters: {'solver': 'saga', 'l1_ratio': 0.12169985218273383, 'C': 46.055747022269266}. Best is trial 3 with value: 0.6321202525997326.
[I 2026-06-05 20:06:59,630] Trial 4 finished with value: 0.6321273760378767 and parameters: {'solver': 'saga', 'l1_ratio': 0.

In [12]:
print("Best score for LR:", study_LR.best_value)
print("Best params for LR:", study_LR.best_params)

Best score for LR: 0.632291841475064
Best params for LR: {'solver': 'saga', 'l1_ratio': 0.2655307249449733, 'C': 35.4685386786118}


In [13]:
# Fitting Logistic Regression pipeline with the best trial
pipe_LR=build_pipeline(trial=study_LR.best_trial, model= 'Logistic Regression')
fitted_pipe_LR=pipe_LR.fit(X_train, y_train)

### K-Nearest Neighbors

In [14]:
# Creating optuna study for K-Nearest Neighbors
study_KNN = optuna.create_study(direction="maximize")

[I 2026-06-05 20:08:42,582] A new study created in memory with name: no-name-61793b35-c305-415b-8c91-39812d350f19


In [15]:
# Optimizing optuna study for K-Nearest Neighbors
study_KNN.optimize(lambda trial: objective(trial, X_train, y_train, model='K-Nearest Neighbors'),n_trials=100)

[I 2026-06-05 20:08:42,871] Trial 0 finished with value: 0.5882745555161903 and parameters: {'knn_n_neighbors': 23, 'knn_weights': 'uniform', 'knn_metric': 'euclidean', 'knn_p': 3}. Best is trial 0 with value: 0.5882745555161903.
[I 2026-06-05 20:08:43,112] Trial 1 finished with value: 0.5957028525179459 and parameters: {'knn_n_neighbors': 35, 'knn_weights': 'uniform', 'knn_metric': 'manhattan', 'knn_p': 2}. Best is trial 1 with value: 0.5957028525179459.
[I 2026-06-05 20:08:43,361] Trial 2 finished with value: 0.5828092979564918 and parameters: {'knn_n_neighbors': 36, 'knn_weights': 'distance', 'knn_metric': 'manhattan', 'knn_p': 3}. Best is trial 1 with value: 0.5957028525179459.
[I 2026-06-05 20:08:45,290] Trial 3 finished with value: 0.5887333958977077 and parameters: {'knn_n_neighbors': 50, 'knn_weights': 'distance', 'knn_metric': 'minkowski', 'knn_p': 3}. Best is trial 1 with value: 0.5957028525179459.
[I 2026-06-05 20:08:47,198] Trial 4 finished with value: 0.5929686325578197 an

In [16]:
print("Best score for KNN:", study_KNN.best_value)
print("Best params for KNN:", study_KNN.best_params)

Best score for KNN: 0.6031276199829867
Best params for KNN: {'knn_n_neighbors': 47, 'knn_weights': 'uniform', 'knn_metric': 'manhattan', 'knn_p': 1}


In [17]:
# Fitting K-Nearest Neighbors pipeline with the best trial
pipe_KNN=build_pipeline(trial=study_KNN.best_trial, model= 'K-Nearest Neighbors')
fitted_pipe_KNN=pipe_KNN.fit(X_train, y_train)

### Support Vector Machine

In [18]:
# Creating optuna study for Support Vector Machine
study_SVM = optuna.create_study(direction="maximize")

[I 2026-06-05 19:47:54,790] A new study created in memory with name: no-name-180d694f-a739-4ec2-a529-07f1bd4cc500


In [19]:
# Optimizing optuna study for Support Vector Machine
study_SVM.optimize(lambda trial: objective(trial, X_train, y_train, model='Support Vector Machine'),n_trials=100)

[I 2026-06-05 19:47:57,085] Trial 0 finished with value: 0.596526081066813 and parameters: {'svc_C': 0.045226555476095345, 'svc_kernel': 'linear', 'svc_gamma': 6.4627416431187e-05, 'svc_degree': 4}. Best is trial 0 with value: 0.596526081066813.
[I 2026-06-05 19:48:02,080] Trial 1 finished with value: 0.6074161706691956 and parameters: {'svc_C': 0.002449641166797178, 'svc_kernel': 'rbf', 'svc_gamma': 0.04536150494034136, 'svc_degree': 2}. Best is trial 1 with value: 0.6074161706691956.
[I 2026-06-05 19:48:16,768] Trial 2 finished with value: 0.5233796065965034 and parameters: {'svc_C': 0.007682607615114961, 'svc_kernel': 'poly', 'svc_gamma': 1.28925952540971, 'svc_degree': 4}. Best is trial 1 with value: 0.6074161706691956.
[I 2026-06-05 19:48:19,296] Trial 3 finished with value: 0.621037937087001 and parameters: {'svc_C': 0.0012656583498744595, 'svc_kernel': 'poly', 'svc_gamma': 0.3364510615205489, 'svc_degree': 2}. Best is trial 3 with value: 0.621037937087001.
[I 2026-06-05 19:48:24

In [ ]:
print("Best score for SVM:", study_SVM.best_value)
print("Best params for SVM:", study_SVM.best_params)

In [ ]:
# Fitting Support Vector Machine pipeline with the best trial
pipe_SVM=build_pipeline(trial=study_SVM.best_trial, model= 'Support Vector Machine')
fitted_pipe_SVM=pipe_SVM.fit(X_train, y_train)

### Decision Tree

In [18]:
# Creating optuna study for Decision Tree
study_DT = optuna.create_study(direction="maximize")

[I 2026-06-05 20:09:05,906] A new study created in memory with name: no-name-466b2e90-a040-43e0-8d03-2300888535bb


In [19]:
# Optimizing optuna study for Decision Tree
study_DT.optimize(lambda trial: objective(trial, X_train, y_train, model='Decision Tree'),n_trials=100)

[I 2026-06-05 20:09:05,984] Trial 0 finished with value: 0.5311821719859315 and parameters: {'dt_criterion': 'entropy', 'dt_max_depth': 26, 'dt_min_samples_split': 3, 'dt_min_samples_leaf': 2, 'dt_max_features': 'log2'}. Best is trial 0 with value: 0.5311821719859315.
[I 2026-06-05 20:09:06,043] Trial 1 finished with value: 0.5861876790961112 and parameters: {'dt_criterion': 'gini', 'dt_max_depth': 11, 'dt_min_samples_split': 5, 'dt_min_samples_leaf': 8, 'dt_max_features': 'log2'}. Best is trial 1 with value: 0.5861876790961112.
[I 2026-06-05 20:09:06,121] Trial 2 finished with value: 0.5537395958977991 and parameters: {'dt_criterion': 'gini', 'dt_max_depth': 45, 'dt_min_samples_split': 9, 'dt_min_samples_leaf': 5, 'dt_max_features': None}. Best is trial 1 with value: 0.5861876790961112.
[I 2026-06-05 20:09:06,195] Trial 3 finished with value: 0.5813149457838367 and parameters: {'dt_criterion': 'gini', 'dt_max_depth': 17, 'dt_min_samples_split': 13, 'dt_min_samples_leaf': 4, 'dt_max_fe

In [20]:
print("Best score for DT:", study_DT.best_value)
print("Best params for DT:", study_DT.best_params)

Best score for DT: 0.6196342157326636
Best params for DT: {'dt_criterion': 'gini', 'dt_max_depth': 6, 'dt_min_samples_split': 14, 'dt_min_samples_leaf': 10, 'dt_max_features': None}


In [21]:
# Fitting Decision Tree pipeline with the best trial
pipe_DT=build_pipeline(trial=study_DT.best_trial, model= 'Decision Tree')
fitted_pipe_DT=pipe_DT.fit(X_train, y_train)

### Random Forest

In [22]:
# Creating optuna study for Random Forest
study_RF = optuna.create_study(direction="maximize")

[I 2026-06-05 20:09:13,536] A new study created in memory with name: no-name-e2409c5a-203b-4c0a-9cff-83eeefba3126


In [23]:
# Optimizing optuna study for Random Forest
study_RF.optimize(lambda trial: objective(trial, X_train, y_train, model='Random Forest'),n_trials=100)

[I 2026-06-05 20:09:14,565] Trial 0 finished with value: 0.6317903892881646 and parameters: {'rf_n_estimators': 500, 'rf_criterion': 'entropy', 'rf_max_depth': 24, 'rf_min_samples_split': 2, 'rf_min_samples_leaf': 4, 'rf_max_features': 'log2', 'rf_bootstrap': True}. Best is trial 0 with value: 0.6317903892881646.
[I 2026-06-05 20:09:15,380] Trial 1 finished with value: 0.6358178356195292 and parameters: {'rf_n_estimators': 400, 'rf_criterion': 'entropy', 'rf_max_depth': 31, 'rf_min_samples_split': 20, 'rf_min_samples_leaf': 9, 'rf_max_features': 'sqrt', 'rf_bootstrap': True}. Best is trial 1 with value: 0.6358178356195292.
[I 2026-06-05 20:09:16,875] Trial 2 finished with value: 0.6263526548837886 and parameters: {'rf_n_estimators': 400, 'rf_criterion': 'gini', 'rf_max_depth': 17, 'rf_min_samples_split': 2, 'rf_min_samples_leaf': 8, 'rf_max_features': None, 'rf_bootstrap': True}. Best is trial 1 with value: 0.6358178356195292.
[I 2026-06-05 20:09:18,098] Trial 3 finished with value: 0.

In [24]:
print("Best score for RF:", study_RF.best_value)
print("Best params for RF:", study_RF.best_params)

Best score for RF: 0.6387040563615111
Best params for RF: {'rf_n_estimators': 100, 'rf_criterion': 'entropy', 'rf_max_depth': 32, 'rf_min_samples_split': 20, 'rf_min_samples_leaf': 5, 'rf_max_features': 'sqrt', 'rf_bootstrap': True}


In [25]:
# Fitting Random Forest pipeline with the best trial
pipe_RF=build_pipeline(trial=study_RF.best_trial, model= 'Random Forest')
fitted_pipe_RF=pipe_RF.fit(X_train, y_train)

### Model performances

In [26]:
# Evaluate model performance
metrics_XGB=evaluate_model(fitted_pipe_XGB,X_test,y_test)
print(metrics_XGB['report'])

2026-06-05 20:11:14,522 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for XGBClassifier classification model.


              precision    recall  f1-score   support

   Not Churn       0.89      0.82      0.85      1033
       Churn       0.59      0.71      0.64       374

    accuracy                           0.79      1407
   macro avg       0.74      0.77      0.75      1407
weighted avg       0.81      0.79      0.80      1407



In [ ]:
metrics_LGBM=evaluate_model(fitted_pipe_LGBM,X_test,y_test)
print(metrics_LGBM['report'])

[LightGBM] [Info] Number of positive: 1196, number of negative: 3304
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.166663 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 657
[LightGBM] [Info] Number of data points in the train set: 4500, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> initscore=-1.016151
[LightGBM] [Info] Start training from score -1.016151
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [27]:
metrics_LR=evaluate_model(fitted_pipe_LR,X_test,y_test)
print(metrics_LR['report'])

2026-06-05 20:11:14,559 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for LogisticRegression classification model.


              precision    recall  f1-score   support

   Not Churn       0.91      0.74      0.81      1033
       Churn       0.52      0.79      0.63       374

    accuracy                           0.75      1407
   macro avg       0.72      0.77      0.72      1407
weighted avg       0.81      0.75      0.77      1407



In [28]:
metrics_KNN=evaluate_model(fitted_pipe_KNN,X_test,y_test)
print(metrics_KNN['report'])

2026-06-05 20:11:14,716 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for KNeighborsClassifier classification model.


              precision    recall  f1-score   support

   Not Churn       0.86      0.88      0.87      1033
       Churn       0.64      0.60      0.62       374

    accuracy                           0.81      1407
   macro avg       0.75      0.74      0.75      1407
weighted avg       0.80      0.81      0.80      1407



In [ ]:
metrics_SVM=evaluate_model(fitted_pipe_SVM,X_test,y_test)
print(metrics_SVM['report'])

In [29]:
metrics_DT=evaluate_model(fitted_pipe_DT,X_test,y_test)
print(metrics_DT['report'])

2026-06-05 20:11:14,750 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for DecisionTreeClassifier classification model.


              precision    recall  f1-score   support

   Not Churn       0.90      0.73      0.81      1033
       Churn       0.51      0.78      0.62       374

    accuracy                           0.75      1407
   macro avg       0.71      0.76      0.71      1407
weighted avg       0.80      0.75      0.76      1407



In [30]:
metrics_RF=evaluate_model(fitted_pipe_RF,X_test,y_test)
print(metrics_RF['report'])

2026-06-05 20:11:14,843 | INFO | evaluate_model | Calculated accuracy, roc_auc, f_1, precision, recall, average_precision, confusion_matrix and report for RandomForestClassifier classification model.


              precision    recall  f1-score   support

   Not Churn       0.89      0.78      0.83      1033
       Churn       0.54      0.74      0.63       374

    accuracy                           0.77      1407
   macro avg       0.72      0.76      0.73      1407
weighted avg       0.80      0.77      0.78      1407

